# 🚀 医学图像处理流水线 - 5分钟快速演示

本notebook展示完整的医学图像处理流程：
1. ✅ 加载测试数据
2. ✅ 可视化原始数据
3. ✅ 运行Marching Cubes三维重建
4. ✅ 交互式3D可视化

**运行方式**: 按 `Shift+Enter` 逐个运行单元格

## 📦 步骤1: 安装和导入依赖

In [ ]:
# 安装必要的包（首次运行需要）
!pip install numpy plotly matplotlib -q

import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess
import os

print("✅ 依赖导入成功！")

## 📊 步骤2: 加载和可视化测试数据

In [ ]:
# 加载NPY数据
data_path = Path('marching_cubes_c/case_00000_x.npy')

if not data_path.exists():
    print(f"❌ 数据文件不存在: {data_path}")
    print("请确保在项目根目录运行此notebook")
else:
    volume = np.load(data_path)
    print(f"✅ 数据加载成功！")
    print(f"   数据形状: {volume.shape}")
    print(f"   数据类型: {volume.dtype}")
    print(f"   数值范围: [{volume.min():.2f}, {volume.max():.2f}]")
    print(f"   数据大小: {volume.nbytes / 1024:.2f} KB")

In [ ]:
# 可视化几个切片
if volume.ndim == 4:
    volume = volume[0]  # 如果是4D，取第一个通道

depth = volume.shape[0]
slice_indices = [depth//4, depth//2, 3*depth//4]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('医学图像切片预览', fontsize=16, fontweight='bold')

for idx, slice_idx in enumerate(slice_indices):
    axes[idx].imshow(volume[slice_idx], cmap='gray')
    axes[idx].set_title(f'切片 {slice_idx}/{depth}', fontsize=12)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print("✅ 切片可视化完成！")

## 🔧 步骤3: 检查Marching Cubes可执行文件

In [ ]:
# 检查可执行文件是否存在
exe_paths = [
    Path('marching_cubes_c/build/Release/marching_cubes_c.exe'),
    Path('marching_cubes_c/build/marching_cubes_c.exe'),
    Path('marching_cubes_c/build/Debug/marching_cubes_c.exe')
]

exe_path = None
for path in exe_paths:
    if path.exists():
        exe_path = path
        break

if exe_path:
    print(f"✅ 找到可执行文件: {exe_path}")
else:
    print("❌ 未找到可执行文件，需要先编译项目")
    print("\n请运行以下命令编译:")
    print("  cd marching_cubes_c")
    print("  mkdir build && cd build")
    print("  cmake ..")
    print("  cmake --build . --config Release")
    print("\n或者直接运行: run_full_demo.bat")

## 🎯 步骤4: 运行Marching Cubes算法

**注意**: 如果上一步未找到可执行文件，请先编译项目

In [ ]:
if exe_path and exe_path.exists():
    # 设置参数
    input_npy = 'marching_cubes_c/case_00000_x.npy'
    output_vtk = 'marching_cubes_c/demo_output.vtk'
    iso_value = 0.5
    
    # 构建命令
    cmd = [
        str(exe_path),
        '--input', input_npy,
        '--iso', str(iso_value),
        '--vtk', output_vtk
    ]
    
    print(f"🚀 正在运行Marching Cubes算法...")
    print(f"   等值: {iso_value}")
    print(f"   输出: {output_vtk}")
    print()
    
    # 运行命令
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ Marching Cubes运行成功！")
        print(result.stdout)
        vtk_file = output_vtk
    else:
        print("❌ 运行失败")
        print(result.stderr)
        vtk_file = None
else:
    # 尝试使用已存在的VTK文件
    if Path('marching_cubes_c/out.vtk').exists():
        print("ℹ️  使用已存在的VTK文件: marching_cubes_c/out.vtk")
        vtk_file = 'marching_cubes_c/out.vtk'
    else:
        print("❌ 无法运行Marching Cubes，也没有找到已有的VTK文件")
        vtk_file = None

## 🎨 步骤5: 读取VTK文件

In [ ]:
def read_vtk_manual(vtk_path):
    """手动解析VTK Legacy格式文件"""
    vertices = []
    triangles = []
    
    with open(vtk_path, 'r') as f:
        lines = f.readlines()
        
        # 查找POINTS部分
        i = 0
        while i < len(lines):
            if 'POINTS' in lines[i]:
                num_points = int(lines[i].split()[1])
                i += 1
                for _ in range(num_points):
                    coords = list(map(float, lines[i].split()))
                    vertices.append(coords[:3])
                    i += 1
                break
            i += 1
        
        # 查找POLYGONS部分
        while i < len(lines):
            if 'POLYGONS' in lines[i]:
                num_polys = int(lines[i].split()[1])
                i += 1
                for _ in range(num_polys):
                    poly_data = list(map(int, lines[i].split()))
                    if poly_data[0] == 3:  # 三角形
                        triangles.append(poly_data[1:4])
                    i += 1
                break
            i += 1
    
    return np.array(vertices), np.array(triangles)

if vtk_file and Path(vtk_file).exists():
    print(f"📖 正在读取VTK文件: {vtk_file}")
    vertices, triangles = read_vtk_manual(vtk_file)
    
    print(f"✅ VTK文件读取成功！")
    print(f"   顶点数量: {len(vertices):,}")
    print(f"   三角形数量: {len(triangles):,}")
    print(f"   X范围: [{vertices[:, 0].min():.2f}, {vertices[:, 0].max():.2f}]")
    print(f"   Y范围: [{vertices[:, 1].min():.2f}, {vertices[:, 1].max():.2f}]")
    print(f"   Z范围: [{vertices[:, 2].min():.2f}, {vertices[:, 2].max():.2f}]")
else:
    print("❌ 没有可用的VTK文件")
    vertices = None
    triangles = None

## 🌟 步骤6: 交互式3D可视化

**提示**: 
- 🖱️ 拖动旋转模型
- 🔍 滚轮缩放
- 🔄 双击重置视图

In [ ]:
if vertices is not None and triangles is not None:
    # 翻转Z轴（医学图像常用）
    vertices[:, 2] = -vertices[:, 2]
    
    # 计算顶点深度用于颜色映射
    z_values = vertices[:, 2]
    
    # 创建3D mesh
    fig = go.Figure(data=[
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=triangles[:, 0],
            j=triangles[:, 1],
            k=triangles[:, 2],
            intensity=z_values,
            colorscale='Viridis',
            showscale=True,
            lighting=dict(
                ambient=0.4,
                diffuse=0.8,
                specular=0.5,
                roughness=0.3,
                fresnel=0.2
            ),
            lightposition=dict(x=100, y=200, z=0),
            flatshading=False,
            opacity=1.0,
            name='3D重建模型'
        )
    ])
    
    # 设置布局
    fig.update_layout(
        title=dict(
            text='🏥 医学图像三维重建 - 交互式可视化',
            x=0.5,
            xanchor='center',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title='X轴'),
            yaxis=dict(title='Y轴'),
            zaxis=dict(title='Z轴'),
            aspectmode='data',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    fig.show()
    
    print("\n✅ 3D可视化完成！")
    print("\n💡 提示:")
    print("  • 使用鼠标拖动旋转模型")
    print("  • 使用滚轮缩放")
    print("  • 双击重置视图")
else:
    print("❌ 无法创建可视化，请确保前面的步骤都成功执行")

## 🎉 完成！

### 下一步探索

1. **尝试不同的等值**: 修改步骤4中的 `iso_value` 参数
2. **查看更多演示**: 
   - `jupyter_lab/ai_liver/ct_3d_reconstruction_clean.ipynb` - CT完整流程
   - `jupyter_lab/reconstruction/plotly_vtk_render.ipynb` - 高级可视化
3. **AI分割**: 查看 `ai_segmentation/code2/` 目录
4. **图像预处理**: 查看 `jupyter_lab/preprocess/` 目录

### 项目文档
- 📖 主README: `README.md`
- 🚀 快速指南: `quick_demo.md`

---

**遇到问题?** 请检查:
1. Python环境是否正确安装
2. 是否在项目根目录运行
3. Marching Cubes是否已编译